# UniFormer-S + trọng số Kinetics — tái lập đội **hạng 2** LLD-MMRI 2023

Nguồn: [`ZHEGG/miccai2023`](https://github.com/ZHEGG/miccai2023) ·
[leaderboard official](https://github.com/LMMMEng/LLD-MMRI2023/blob/main/assets/test_leaderboard.md)

⚠️ **Đây là hạng 2 (`NPUBXY`, 0.8078), không phải hạng 1** (`WorkingisAllyouneed`, 0.8322).
README của repo tự ghi "second-place solution". Đừng viết nhầm trong báo cáo.

## Vì sao hướng này đáng chi, khi bảy hướng khác đã bị loại

**Baseline official của challenge CHÍNH LÀ UniFormer-S 3D, from scratch → 0.6083.**
Repo này dùng **đúng kiến trúc đó** và bật `--pretrained` (Kinetics-400 video):

| | macro-F1 test-104 |
|---|---|
| UniFormer-S, **from scratch** (baseline official) | 0.6083 |
| UniFormer + **Kinetics** + cb_loss + sqrt sampling + smoothing + drop-path + 3 aug lọc | **0.8078** |

Chẩn đoán §5 của `AGENTS.md` **không loại** hướng này: bảy hướng bị loại đều là chỉnh
loss/ngưỡng/augment **trên cùng một biểu diễn**, còn §4 nói ràng buộc chính *là* biểu diễn
(di căn không vào nổi top-2). Pretrained là can thiệp duy nhất đổi được biểu diễn.

⚠️ Chênh ~0.20 đó **không phải phép thử một biến sạch** — nó gộp 6 thứ. Ta tái lập cả cụm
và chỉ quy kết được cho **cả cụm**.

## Cần mount gì

| | |
|---|---|
| **Cache CGHNet** (`128×128×16`) | Dùng LẠI, không build mới. `--img_size 16 128 128 --crop_size 14 112 112` của họ khớp chính xác |
| **Internet: BẬT** | để tải `uniformer_small_k400_16x8.pth` (~200 MB) từ HuggingFace |

Tải xong nên lưu thành Kaggle Dataset và mount cho session sau (AGENTS.md §7).

## Năm cổng chạy TRƯỚC khi cam kết fold nào

| cổng | bắt lỗi kiểu gì | tiền lệ |
|---|---|---|
| **A** trọng số | khoá **nào** thiếu, không phải bao nhiêu % | E8: sai `shortcut_type` mà vẫn khớp 85% (S-118) |
| **B** hình học | shape thật + số token từng stage | E2: chạy 48 in-plane suốt, không gì báo (S-065) |
| **C** ngân sách | s/epoch **đo thật** | E13: phát hiện 79 s/epoch *sau khi* đã cam kết (S-120) |
| **D** sampler | `sqrt` có thật sự kéo lớp hiếm lên không | khoá mới, chưa có tiền lệ |
| **E** augment | ba phép lọc có áp **giống nhau cho 8 pha** không | E6: xáo độc lập từng pha ⇒ ICC −0.085, di căn −0.111 (S-102) |

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2]                 # sàng 2 fold. ⚠️ 2 fold chỉ để LOẠI, không để CHỌN (S-107)
CONFIG_NAME = "uniformer_s.yaml"
PREPROCESS_NAME = "preprocess_cghnet.yaml"
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
M = CFG["model"]

INNER = tuple(PRE["target_size"])                       # [X, Y, Z] = 112, 112, 14
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))  # 128, 128, 16
assert tuple(CFG["data"]["crop_size"]) == INNER, "data.crop_size lệch target_size của cache"

# Thứ tự của MODEL là (D, H, W) với D = trục lát; cache là [X, Y, Z]. Đổi một lần ở đây và
# dùng biến này ở mọi cổng, để không ai phải nhẩm lại giữa chừng.
MODEL_SIZE = (INNER[2], INNER[0], INNER[1])             # (14, 112, 112)

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"model     : uniformer {M['variant']} · patch_embed1_stride {M['patch_embed1_stride']}")
print(f"            drop_path {M['drop_path_rate']} · drop_rate {M['drop_rate']} · "
      f"head_dropout {M['head_dropout']}")
print(f"loss      : {CFG['loss']['name']} gamma {CFG['loss']['gamma']} · "
      f"class_weights {CFG['loss']['class_weights']} · smoothing {CFG['loss']['label_smoothing']}")
print(f"sampler   : {CFG['data']['sampling']}")
print(f"augment   : edge {CFG['data']['augment']['edge_prob']} · "
      f"emboss {CFG['data']['augment']['emboss_prob']} · "
      f"filter {CFG['data']['augment']['filter_prob']}")
print(f"hình học  : cache {GRID} -> model nhận {INNER} [X,Y,Z] = {MODEL_SIZE} (D,H,W)")

## 1. Cache CGHNet — dùng lại, không build mới

`--img_size 16 128 128 --crop_size 14 112 112` của họ khớp **chính xác**
`configs/preprocess_cghnet.yaml`. Cache E4 (`112×112×32`) và E12 (`136×136×40`) **không**
dùng được.

In [ ]:
import json as _json

import numpy as np

CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    khop = all(meta.get(k) == v for k, v in CAN.items())
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ CGHNet' if khop else '  --    '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache CGHNet.\n"
        f"  Cần cache có {CAN}\n"
        "  Chạy notebooks/18_build_cache_cghnet.ipynb trước (CPU, Accelerator = None,\n"
        "  ~20 phút), lưu output thành Dataset, rồi mount vào đây.\n"
        "  Cache E4 và cache E12 KHÔNG dùng được: hình học khác hẳn."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"
with np.load(next(CACHE_DIR.glob("*.npz"))) as z:
    shape = tuple(z["image"].shape)
assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — đây không phải cache CGHNet"
print(f"\ncache ✓ · {CACHE_DIR} · {n_npz} ca · mảng {shape}")

## 1b. Trọng số Kinetics

`uniformer_small_k400_16x8.pth` — **đúng file** repo hạng 2 dùng cho `uniformer_small_original`.
Ưu tiên bản đã mount; không có thì tải từ HuggingFace (cần **bật Internet**).

In [ ]:
import urllib.request

from src.models.uniformer3d import HF_BASE_URL, PRETRAINED_FILENAMES

FILENAME = PRETRAINED_FILENAMES[M["variant"]]

# 1) Bản đã mount làm Dataset — không tốn băng thông, và dùng được cả khi tắt Internet.
CKPT = next(iter(sorted(Path("/kaggle/input").rglob(FILENAME))), None)

# 2) Không có thì tải.
if CKPT is None:
    CKPT = Path("/kaggle/working/pretrained") / FILENAME
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    if not CKPT.exists():
        url = HF_BASE_URL + FILENAME
        print("tải:", url)
        try:
            urllib.request.urlretrieve(url, CKPT)
        except Exception as exc:  # noqa: BLE001 - cần thông báo dạy được, không phải traceback
            raise RuntimeError(
                f"Không tải được {FILENAME}. Hai nguyên nhân thường gặp:\n"
                "  1. Internet đang TẮT — bật ở panel bên phải (Settings > Internet).\n"
                "  2. HuggingFace chặn tạm — tải tay rồi upload thành Kaggle Dataset,\n"
                f"     cell này tự tìm theo tên file {FILENAME} dưới /kaggle/input.\n"
                f"  URL: {url}"
            ) from exc

os.environ["LLDMMRI_PRETRAINED_PATH"] = str(CKPT)
print(f"trọng số ✓ · {CKPT} · {CKPT.stat().st_size / 1e6:.0f} MB")

## Cổng A ⚠️⚠️ — trọng số có thật sự vào model không

**`load_state_dict(strict=False)` không báo lỗi khi không khoá nào khớp.** Model vẫn chạy,
vẫn ra số — chỉ là "có pretrained" lặng lẽ thành "không pretrained", và cả thí nghiệm này tồn
tại để đo đúng biến đó.

⚠️ Cổng này in **khoá NÀO** thiếu, không in tỉ lệ phần trăm. E8 đặt sai `shortcut_type` mà tỉ
lệ khớp vẫn ~85% — dư sức qua một ngưỡng 50% (WORKLOG S-118).

Chỉ hai tiền tố được phép thiếu: `patch_embed1.` (8 kênh MRI ≠ 3 kênh RGB) và `head.`
(7 lớp ≠ 400 lớp Kinetics). Repo hạng 2 cũng bỏ đúng hai cái đó.

In [ ]:
import torch

from src.models.uniformer3d import DROPPED_PREFIXES, build_uniformer3d, load_kinetics_weights

tham_so = {k: v for k, v in M.items() if k not in ("name", "require_pretrained")}
tham_so["pretrained_path"] = None
model = build_uniformer3d(**tham_so, require_pretrained=False)

n_tham_so = sum(p.numel() for p in model.parameters())
bao_cao = load_kinetics_weights(model, CKPT)   # NỔ nếu thiếu khoá ngoài hai tiền tố trên

print(f"tham số model : {n_tham_so / 1e6:.2f}M")
print(f"khoá nạp được : {len(bao_cao['loaded'])}")
print(f"khoá thiếu    : {len(bao_cao['missing'])}  (chỉ được phép {DROPPED_PREFIXES})")
for k in bao_cao["missing"]:
    print("   thiếu :", k)
print(f"khoá dư trong checkpoint: {len(bao_cao['unexpected'])}")
for k in bao_cao["unexpected"][:10]:
    print("   dư    :", k)

assert bao_cao["loaded"], "⛔ KHÔNG nạp được khoá nào — đây là run from scratch trá hình"
assert all(k.startswith(DROPPED_PREFIXES) for k in bao_cao["missing"])
# Một mạng UniFormer-S có ~300 khoá; nạp được dưới 100 nghĩa là khớp nhầm nhánh nào đó.
assert len(bao_cao["loaded"]) > 100, f"chỉ nạp {len(bao_cao['loaded'])} khoá — quá ít"
print("\ncổng A ✓ — trọng số Kinetics ĐÃ vào model")

## Cổng B ⚠️⚠️⚠️ — hình dạng thật qua từng stage

E2 chạy suốt ở 48 in-plane thay vì 96 và **không có gì báo** (WORKLOG S-065). Cell này bắt
hook vào từng `patch_embed` để đọc shape **thật**, rồi đối chiếu với `stage_token_counts`
tính bằng tay.

⚠️ Đây cũng là chỗ đọc ra nút thắt ngân sách: `patch_embed1` stride `(1,2,2)` **không hạ mẫu
trục lát**, nên stage 3 có **2744** token so với **1568** của bản pretrained.

In [ ]:
from src.models.uniformer3d import stage_token_counts

thuc_te = []
hooks = [
    getattr(model, f"patch_embed{i}").register_forward_hook(
        lambda _m, _i, out: thuc_te.append(tuple(out.shape[2:]))
    )
    for i in range(1, 5)
]
model.eval()
with torch.no_grad():
    ra = model(torch.zeros(1, 8, *INNER))
for h in hooks:
    h.remove()

tinh_tay = stage_token_counts(MODEL_SIZE, M["patch_embed1_stride"])
pretrained = stage_token_counts((16, 224, 224), (2, 4, 4))

print(f"{'stage':<6}{'thực tế (D,H,W)':>20}{'tính tay':>18}{'token':>10}{'pretrained':>12}")
for i, (that, tay, pre) in enumerate(zip(thuc_te, tinh_tay, pretrained), start=1):
    loai = "CBlock" if i <= 2 else "SABlock"
    print(f"{i} {loai:<5}{str(that):>18}{str(tay):>18}"
          f"{that[0] * that[1] * that[2]:>10}{pre[0] * pre[1] * pre[2]:>12}")

assert thuc_te == tinh_tay, f"⛔ shape thật {thuc_te} lệch tính tay {tinh_tay}"
assert tuple(ra.shape) == (1, CFG["model"]["num_classes"])

n3 = tinh_tay[2][0] * tinh_tay[2][1] * tinh_tay[2][2]
n3_pre = pretrained[2][0] * pretrained[2][1] * pretrained[2][2]
print(f"\nstage 3 (attention TOÀN CỤC, depth 8): {n3} token so với {n3_pre} của bản pretrained"
      f"  ⇒ {n3 / n3_pre:.2f}× token, ~{(n3 / n3_pre) ** 2:.1f}× chi phí attention")
print(f"đầu ra: {tuple(ra.shape)}")
print("\ncổng B ✓")

## Cổng C ⚠️ — ngân sách, ĐO THẬT

**E13 mất cả một session vì phát hiện 79 s/epoch sau khi đã cam kết** (WORKLOG S-120).
Và AGENTS.md §6 ghi rõ: **không suy giờ từ GFLOPs** — ước lượng kiểu đó cho CGHNet đã sai
xa (đoán ~8h/fold, thật 1.6h).

Quá **60 s/epoch** thì đổi `model.patch_embed1_stride` sang `[2, 2, 2]` (lát 14→7, còn 1372
token — dưới cả bản pretrained) rồi chạy lại cell này.

In [ ]:
import time

from src.train.run import build_loaders

train_loader, val_loader, train_labels = build_loaders(CFG, FOLDS[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).train()
opt = torch.optim.AdamW(model.parameters(), lr=float(CFG["train"]["lr"]))
scaler = torch.amp.GradScaler("cuda", enabled=bool(CFG["train"]["amp"]))

N_DO = 8
t0 = None
for i, batch in enumerate(train_loader):
    if i == 2:          # bỏ 2 batch đầu: nạp worker + cudnn benchmark
        torch.cuda.synchronize() if device.type == "cuda" else None
        t0 = time.perf_counter()
    if i == 2 + N_DO:
        break
    x = batch["image"].to(device, non_blocking=True)
    y = batch["label"].to(device, non_blocking=True)
    with torch.amp.autocast("cuda", enabled=bool(CFG["train"]["amp"])):
        loss = torch.nn.functional.cross_entropy(model(x), y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()

giay_moi_batch = (time.perf_counter() - t0) / N_DO
n_batch = len(train_loader)
s_epoch = giay_moi_batch * n_batch * 1.15      # +15% cho vòng val, ước từ các run trước
gio_fold = s_epoch * int(CFG["train"]["epochs"]) / 3600

print(f"thiết bị      : {device}")
print(f"batch train   : {n_batch} · batch_size {CFG['data']['batch_size']}")
print(f"giây/batch    : {giay_moi_batch:.3f}")
print(f"giây/epoch    : {s_epoch:.1f}   (đã cộng ~15% cho val)")
print(f"giờ/fold      : {gio_fold:.2f}  ({CFG['train']['epochs']} epoch)")
print(f"giờ/{len(FOLDS)} fold  : {gio_fold * len(FOLDS):.2f}")
print(f"giờ/5 fold    : {gio_fold * 5:.2f}")
print("\nđối chiếu: CGHNet 1.6 h/fold (đo thật) · E4 3.8 h/fold · session Kaggle tối đa 12h")

if s_epoch > 60:
    print(f"\n⛔ {s_epoch:.0f} s/epoch — QUÁ NGƯỠNG 60.")
    print("   Đổi model.patch_embed1_stride sang [2, 2, 2] trong configs/uniformer_s.yaml")
    print("   (lát 14 -> 7, stage 3 còn 1372 token) rồi chạy lại từ cổng A.")
else:
    print("\ncổng C ✓")

model.cpu()
del opt, scaler
torch.cuda.empty_cache() if device.type == "cuda" else None

## Cổng D ⚠️ — `--sampling sqrt` có thật sự kéo lớp hiếm lên không

⚠️ **Xung đột đã biết với chẩn đoán §1.** Recipe của họ bật *đồng thời* `--cb_loss` (trọng số
lớp trong loss) **và** `--sampling sqrt` (lấy mẫu lại) — **hai lớp cân bằng cùng lúc**. Chẩn
đoán của dự án đo ICC bị dự đoán **thừa** 1.26× và áp-xe 1.31× trên E4, tức đẩy thêm là đi
ngược bằng chứng.

Hai điều đó không mâu thuẫn: §1 đo trên **DenseNet from scratch**, một biểu diễn khác có cán
cân khác. **Tái lập trung thực trước, đo lại sau.** Sau fold 1 hãy chạy
`python -m src.eval.weak_classes --run-dir runs/uniformer_s`; nếu ICC/áp-xe vượt **1.4×** thì
`data.sampling: instance` là ablation **một khoá**, có căn cứ.

In [ ]:
import collections

from src.data.taxonomy import SHORT_NAMES

sampler = train_loader.sampler
print("sampler:", type(sampler).__name__)
assert CFG["data"]["sampling"] != "instance", "config đang ở instance — cổng này vô nghĩa"
assert sampler is not None and hasattr(sampler, "weights"), "⛔ sampler chưa được nối vào"

that = collections.Counter(train_labels)
lay = collections.Counter(np.asarray(train_labels)[list(iter(sampler))])

print(f"\n{'lớp':<10}{'thật':>7}{'được lấy':>11}{'tỉ lệ':>8}")
for c in sorted(SHORT_NAMES):
    print(f"{SHORT_NAMES[c]:<10}{that[c]:>7}{lay[c]:>11}{lay[c] / max(that[c], 1):>8.2f}")

# Không ghim chỉ số lớp: HCC là lớp **6** chứ không phải 0 trong `src/data/taxonomy.py`,
# và ghim nhầm sẽ làm cổng này luôn xanh mà chẳng kiểm gì.
dong, hiem = max(that, key=that.get), min(that, key=that.get)
assert lay[dong] < that[dong], f"⛔ lớp đông nhất ({SHORT_NAMES[dong]}) không bị lấy ít đi"
assert lay[hiem] > that[hiem], f"⛔ lớp hiếm nhất ({SHORT_NAMES[hiem]}) không được lấy nhiều hơn"
print(f"\nđông nhất {SHORT_NAMES[dong]}: {that[dong]} -> {lay[dong]}   "
      f"hiếm nhất {SHORT_NAMES[hiem]}: {that[hiem]} -> {lay[hiem]}")
print("\ncổng D ✓ — sqrt cân bằng MỘT PHẦN, đúng như mong đợi (không cực đoan như `class`)")

## Cổng E ⚠️ — ba augment lọc, và bài học đắt nhất của dự án

**E6 (WORKLOG S-102):** `RandomIntensity` áp scale/shift **độc lập từng pha** làm
**ICC −0.085** và **di căn −0.111**, vì chẩn đoán u gan dựa vào cường độ *tương đối giữa các
pha*. Ba augment mới phải áp **cùng một tham số cho cả 8 pha** — và cell này kiểm điều đó
trên đúng chuỗi transform mà run thật dùng.

Kiểm thêm: rìa khối train và val phải có cùng tỉ lệ voxel 0 (lệch > 0.02 là lỗi kiểu E12).

In [ ]:
from src.data.transforms import RandomAppearance, build_train_transform

chain = build_train_transform(CFG["data"]["augment"], CFG["data"]["crop_size"])
app = [t for t in chain.transforms if isinstance(t, RandomAppearance)]
assert len(app) == 1, f"⛔ có {len(app)} RandomAppearance trong chuỗi, cần đúng 1"
assert isinstance(chain.transforms[-1], RandomAppearance), "⛔ phải nằm CUỐI, sau RandomCrop3D"
print("chuỗi transform:", [type(t).__name__ for t in chain.transforms])

# --- E1: cùng tham số cho 8 pha ---------------------------------------------
from src.utils.seed import set_seed

set_seed(1337)
lech, n_ap = 0, 0
for _ in range(200):
    goc = torch.randn(8, *GRID)
    goc[3] = goc[0]                       # pha 3 sao chép pha 0
    ra = app[0]({"image": goc.clone()})["image"]
    if not torch.allclose(ra, goc):       # có phép nào đó đã áp
        n_ap += 1
        if not torch.allclose(ra[0], ra[3], atol=1e-4):
            lech += 1
print(f"\n200 lượt: {n_ap} lượt bị áp phép ({n_ap / 2:.0f}%, kỳ vọng ~40%) · {lech} lượt lệch pha")
assert lech == 0, f"⛔ {lech} lượt áp khác nhau giữa các pha — đúng lỗi E6, DỪNG"

# --- E2: rìa train so với val -----------------------------------------------
def ti_le_0_o_ria(x):
    ria = torch.cat([x[:, :4].flatten(), x[:, -4:].flatten(),
                     x[:, :, :4].flatten(), x[:, :, -4:].flatten()])
    return float((ria.abs() < 1e-6).float().mean())

tr = np.mean([ti_le_0_o_ria(next(iter(train_loader))["image"][0]) for _ in range(3)])
va = np.mean([ti_le_0_o_ria(next(iter(val_loader))["image"][0]) for _ in range(3)])
print(f"voxel 0 ở rìa: train {tr:.4f} · val {va:.4f} · lệch {abs(tr - va):.4f}")
assert abs(tr - va) < 0.02, "⛔ lệch phân bố rìa train/val — kiểm rotate_mode có phải nearest"
print("\ncổng E ✓")

## 2. Train

Năm cổng phải xanh hết trước khi chạy cell này. `resume: true` nên ngắt session giữa chừng
thì chạy lại là đọc tiếp từ `last.pt`.

In [ ]:
from src.train.run import TRAIN_RESULT_KEYS, train

del model
torch.cuda.empty_cache()

print("khoá train() trả về:", TRAIN_RESULT_KEYS)
results = {}
for fold in FOLDS:
    print(f"\n{'=' * 70}\nfold {fold}\n{'=' * 70}")
    results[fold] = train(CFG_PATH, fold)
    # ⚠️ `best_macro_f1`, KHÔNG phải `macro_f1` — ba tên cho cùng một đại lượng, và đọc sai
    # tên đã làm hỏng một phiên (WORKLOG S-123).
    print("fold %d xong: macro-F1 %.4f @ epoch %d"
          % (fold, results[fold]["best_macro_f1"], results[fold]["best_epoch"]))

## 3. Kết quả từng fold, và bar quyết định đã chốt trước

⚠️ **`/kaggle/working` bị xoá khi session kết thúc.** Chạy mục này ở session khác với session
train thì thư mục run không còn — cell **sẽ nổ kèm hướng dẫn** thay vì in bảng rỗng (bài học
S-124: một cell im lặng trông y như một cell chưa có gì để in).

In [ ]:
import csv as _csv


def _tim_run():
    trong_session = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
    if list(trong_session.glob("fold*/metrics_best.json")):
        return trong_session
    for cand in sorted(Path("/kaggle/input").glob("*/**/fold*/metrics_best.json")):
        print("dùng run đã mount:", cand.parent.parent)
        return cand.parent.parent
    raise RuntimeError(
        "Không thấy fold nào có metrics_best.json. Đã tìm ở "
        + str(trong_session)
        + " và /kaggle/input/**/fold*/. `/kaggle/working` bị xoá khi session kết thúc, nên"
        " nếu bạn train ở session TRƯỚC thì phải upload thư mục run thành Kaggle Dataset rồi"
        " mount vào đây. Hoặc chạy lại cell train ở mục 2 — `resume: true` nên nó đọc tiếp"
        " từ last.pt."
    )


OUT = _tim_run()
E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}   # AGENTS.md §5, CV 5 fold

rows = []
for d in sorted(OUT.glob("fold*")):
    mp = d / "metrics_best.json"
    if not mp.exists():
        continue
    m = _json.loads(mp.read_text("utf-8"))
    fold = int(str(d.name).split("_")[0].replace("fold", ""))
    day = 0
    log = d / "train_log.csv"
    if log.exists():
        vals = [float(r["val_loss"]) for r in _csv.DictReader(log.open(encoding="utf-8"))]
        day = int(np.argmin(vals)) + 1 if vals else 0
    rows.append((fold, m["macro_f1"], m.get("kappa", float("nan")), m.get("epoch", 0), day))

assert rows, "⛔ không đọc được fold nào"

print(f"{'fold':<6}{'macro-F1':>10}{'kappa':>9}{'epoch':>7}{'val_loss đáy':>14}{'E4':>9}{'hiệu':>9}")
for fold, f1, kap, ep, day in sorted(rows):
    print(f"{fold:<6}{f1:>10.4f}{kap:>9.4f}{ep:>7}{day:>14}{E4.get(fold, float('nan')):>9.4f}"
          f"{f1 - E4.get(fold, float('nan')):>+9.4f}")

tb = float(np.mean([r[1] for r in rows]))
tb_e4 = float(np.mean([E4[r[0]] for r in rows if r[0] in E4]))
print(f"\ntrung bình {len(rows)} fold: {tb:.4f}   (E4 cùng fold: {tb_e4:.4f}, "
      f"hiệu {tb - tb_e4:+.4f})")

print("\n" + "=" * 70)
print("BAR QUYẾT ĐỊNH (chốt TRƯỚC khi chạy — AGENTS.md, plan phiên S-125)")
print("=" * 70)
if tb >= 0.78:
    print(f"{tb:.4f} >= 0.78 → pretrained là đòn bẩy THẬT. Chạy đủ 5 fold, đây thành cấu hình chính.")
elif tb >= 0.73:
    print(f"{tb:.4f} trong [0.73, 0.78) → có tác dụng, chưa tới 0.8. Chạy 5 fold, và thử")
    print("   ensemble với E4 ⊕ CGHNet (đa dạng kiến trúc cao ⇒ trùng lặp lỗi nên rất thấp).")
elif tb >= 0.69:
    print(f"{tb:.4f} trong [0.69, 0.72] → NGANG E4. Trọng số Kinetics không chuyển sang MRI")
    print("   đa pha được. DỪNG, và ghi thành kết quả âm có giá trị: ba backbone pretrained")
    print("   độc lập (MedicalNet/E8, Siamese/E13, Kinetics/đây) đều không vượt from-scratch.")
else:
    print(f"{tb:.4f} < 0.69 → NGHI LỖI TRIỂN KHAI hơn là kết luận khoa học (E13 cho <0.5 dù")
    print("   cổng A khớp 102/102 khoá). Đọc lại cổng A và B trước khi kết luận bất cứ điều gì.")

print("\n⚠️ 2 fold chỉ đủ để LOẠI một ý tưởng, KHÔNG đủ để CHỌN nó: E6b cho +0.038 ở 2 fold")
print("   rồi −0.002 ở 5 fold (WORKLOG S-107). Dương ở đây mới chỉ là 'chưa loại được'.")
print("\n⚠️ Đọc F1 của ICC / áp-xe / di căn TRƯỚC macro-F1. Hai dấu hiệu phụ nói cơ chế có")
print("   chạy hay không kể cả khi macro-F1 đứng yên: % lỗi có biên < 0.10 (hiện 0%) và")
print("   top-2 của di căn (hiện BẰNG top-1). Chạy:")
print("     python -m src.eval.weak_classes --run-dir runs/uniformer_s")
print("     python -m src.eval.compare --baseline runs/E4_cv_results --candidate runs/uniformer_s")

## 4. Gói mang về

In [ ]:
import shutil

goi = Path("/kaggle/working") / f"{EXPERIMENT}_results"
if goi.exists():
    shutil.rmtree(goi)
goi.mkdir(parents=True)

for d in sorted(OUT.glob("fold*")):
    dest = goi / d.name
    dest.mkdir()
    for pattern in ("metrics_best.json", "train_log.csv", "val_probs_*.npz", "config.yaml"):
        for f in d.glob(pattern):
            shutil.copy2(f, dest / f.name)

shutil.make_archive(str(goi), "zip", goi)
print("đã gói:", goi.with_suffix(".zip"))
print("⚠️ KHÔNG gói best.pt/last.pt — checkpoint không được commit (AGENTS.md §9), và để")
print("   ensemble hay MC-dropout thì tải riêng từ output của session này.")
for p in sorted(goi.rglob("*")):
    print("  ", p.relative_to(goi))